In [1]:
%pip install pydeseq2

  Using cached numpy-2.5.2-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 114.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 167.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 147.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 65.2 MB/s  0:00:00
Using cached numpy-2.5.2-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.7 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 95.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 123.4 MB/s  0:00:00
  Attempting uninstall: numpym╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  4/27 [pillow]
    Found existing installation: numpy 1.26.4━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  4/27 [pillow]
    Uninstalling numpy-1.26.4:0m╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  5/27 [numpy]low]
      Successfully uninstalled numpy-1.26.4m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  5/27 [numpy]

In [2]:
import os
import pickle as pkl

import numpy as np

from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference
from pydeseq2.ds import DeseqStats
from pydeseq2.utils import load_example_data

In [3]:
counts_df = load_example_data(
    modality="raw_counts",
    dataset="synthetic",
    debug=False,
)

metadata = load_example_data(
    modality="metadata",
    dataset="synthetic",
    debug=False,
)

In [4]:
inference = DefaultInference(n_cpus=8)
dds = DeseqDataSet(
    counts=counts_df,
    metadata=metadata,
    design="~condition",  # compare samples based on the "condition"
    # column ("B" vs "A")
    refit_cooks=True,
    inference=inference,
)

In [5]:
dds.fit_size_factors()

dds.obs["size_factors"]

Using None as control genes, passed at DeseqDataSet initialization


Fitting size factors...
... done in 0.00 seconds.



sample1      1.228981
sample2      1.188774
sample3      0.997222
sample4      1.002158
sample5      0.834577
               ...   
sample96     1.269359
sample97     0.743998
sample98     0.739878
sample99     0.958880
sample100    0.635708
Name: size_factors, Length: 100, dtype: float64

In [6]:
dds.fit_genewise_dispersions()

dds.var["genewise_dispersions"]

Fitting dispersions...
... done in 0.02 seconds.



gene1     0.910135
gene2     0.213718
gene3     0.813594
gene4     0.161362
gene5     0.248504
gene6     0.973077
gene7     0.233030
gene8     0.198103
gene9     0.183636
gene10    0.646375
Name: genewise_dispersions, dtype: float64

In [7]:
dds.fit_dispersion_trend()
dds.uns["trend_coeffs"]
dds.var["fitted_dispersions"]

Fitting dispersion trend curve...
... done in 0.02 seconds.



gene1     0.651424
gene2     0.313001
gene3     1.049865
gene4     0.134145
gene5     0.264005
gene6     0.978128
gene7     0.256765
gene8     0.205750
gene9     0.216026
gene10    0.502746
Name: fitted_dispersions, dtype: float64

In [8]:
dds.fit_dispersion_prior()
print(
    f"logres_prior={dds.uns['_squared_logres']}, sigma_prior={dds.uns['prior_disp_var']}"
)

logres_prior=0.05592493654750395, sigma_prior=0.25


AssertionError: Please provide a fitted DeseqDataSet by first running the `deseq2` method.

In [10]:
dds.fit_MAP_dispersions()
dds.var["MAP_dispersions"]
dds.var["dispersions"]

Fitting MAP dispersions...
... done in 0.03 seconds.



gene1     0.882598
gene2     0.222578
gene3     0.837238
gene4     0.158970
gene5     0.249926
gene6     0.973647
gene7     0.235155
gene8     0.198781
gene9     0.186520
gene10    0.631900
Name: dispersions, dtype: float64

In [11]:
dds.fit_LFC()
dds.varm["LFC"]

Fitting LFCs...
... done in 0.01 seconds.



,Intercept,condition[T.B]
gene1,1.891436,0.438632
gene2,2.851662,0.373296
gene3,1.787780,-0.438645
gene4,4.741958,-0.285647
gene5,3.077798,0.403457
gene6,1.678536,0.001010
gene7,3.291025,0.093116
gene8,3.785129,-0.187604
gene9,3.682882,-0.147443
gene10,2.300515,0.267562


In [12]:
dds.calculate_cooks()
if dds.refit_cooks:
    # Replace outlier counts
    dds.refit()

Calculating cook's distance...
... done in 0.01 seconds.

Replacing 0 outlier genes.



In [13]:
ds = DeseqStats(
    dds,
    contrast=np.array([0, 1]),
    alpha=0.05,
    cooks_filter=True,
    independent_filter=True,
)

In [14]:
ds.run_wald_test()
ds.p_values

Running Wald tests...
... done in 0.01 seconds.



gene1     0.028604
gene2     0.000329
gene3     0.032075
gene4     0.000513
gene5     0.000168
gene6     0.996253
gene7     0.370297
gene8     0.047227
gene9     0.110391
gene10    0.114518
dtype: float64

In [15]:
if ds.cooks_filter:
    ds._cooks_filtering()
ds.p_values

gene1     0.028604
gene2     0.000329
gene3     0.032075
gene4     0.000513
gene5     0.000168
gene6     0.996253
gene7     0.370297
gene8     0.047227
gene9     0.110391
gene10    0.114518
dtype: float64

In [16]:
ds.summary()

Log2 fold change & Wald test p-value, contrast vector: [0 1]
          baseMean  log2FoldChange     lfcSE      stat    pvalue      padj
gene1     8.541317        0.632812  0.289101  2.188898  0.028604  0.064150
gene2    21.281239        0.538552  0.149963  3.591236  0.000329  0.001646
gene3     5.010123       -0.632830  0.295236 -2.143476  0.032075  0.064150
gene4   100.517961       -0.412102  0.118629 -3.473868  0.000513  0.001710
gene5    27.142450        0.582065  0.154706  3.762409  0.000168  0.001646
gene6     5.413043        0.001457  0.310311  0.004696  0.996253  0.996253
gene7    28.294023        0.134338  0.149945  0.895917  0.370297  0.411441
gene8    40.358344       -0.270656  0.136401 -1.984261  0.047227  0.078711
gene9    37.166183       -0.212715  0.133243 -1.596437  0.110391  0.143147
gene10   11.589325        0.386011  0.244588  1.578207  0.114518  0.143147
